# Experimentos del proyecto: M5 Forecast Pipeline

In [1]:
import logging
import time
import warnings

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

# La preparacion y la division vienen del pipeline del proyecto (no se copian aqui).
from pipeline_proyecto import ARCHIVO, NaiveEstacional, cargar_y_dividir, huella, nueva_preparacion

In [2]:
# Base de datos local de MLflow, en esta misma carpeta.
# Para verla: mlflow ui --backend-store-uri sqlite:///mlflow.db
# Silencia los avisos informativos de MLflow para que la salida se lea mejor.
logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENTO = "m5-forecast-pipeline"
mlflow.set_experiment(EXPERIMENTO)
print("Tracking:", mlflow.get_tracking_uri(), "| experimento:", EXPERIMENTO)

Tracking: sqlite:///mlflow.db | experimento: m5-forecast-pipeline


## 1. Datos y división

In [3]:
X_ent, X_prueba, y_ent, y_prueba, division = cargar_y_dividir()
HUELLA = huella()

print(f"Entrenamiento: {division['entrenamiento_desde']} a {division['entrenamiento_hasta']} ({len(X_ent):,} filas)")
print(f"Prueba:        {division['prueba_desde']} a {division['prueba_hasta']} ({len(X_prueba):,} filas)")
print(f"Huella SHA-256 de {ARCHIVO}: {HUELLA}")

Entrenamiento: 2016-04-20 a 2016-05-15 (792,740 filas)
Prueba:        2016-05-16 a 2016-05-22 (213,430 filas)
Huella SHA-256 de proyecto_features.csv: f371b565e51009de51a73291f00f14e7aa541749f6473ee31fea6204d8024abb


## 2. Modelos de la Semana 10

In [4]:
def wape(real, pred):
    real, pred = np.asarray(real), np.asarray(pred)
    return np.abs(real - pred).sum() / np.abs(real).sum()


# nombre de la corrida -> (familia, modelo)
MODELOS = {
    "Naive estacional (igual que hace 7 dias)": ("linea_base", NaiveEstacional()),
    "DummyRegressor (media)": ("linea_base", DummyRegressor(strategy="mean")),
    "Ridge (alpha=1)": ("lineal", Ridge(alpha=1.0)),
    "Ridge (alpha=10000)": ("lineal", Ridge(alpha=10_000.0)),
    "Random Forest sin regularizar": ("ensamblaje", RandomForestRegressor(
        n_estimators=50, max_samples=0.3, n_jobs=-1, random_state=42)),
    "Random Forest regularizado": ("ensamblaje", RandomForestRegressor(
        n_estimators=50, max_samples=0.3, min_samples_leaf=200, max_depth=12,
        n_jobs=-1, random_state=42)),
    "Red neuronal": ("red_neuronal", MLPRegressor(
        hidden_layer_sizes=(64, 32), batch_size=1024, random_state=42)),
}

## 3. Una corrida por modelo

In [5]:
firma = infer_signature(X_ent.head(100), np.zeros(100))
URI_MODELO = {}          # run_id -> direccion del modelo guardado (para registrarlo despues)

for nombre, (familia, modelo) in MODELOS.items():
    # El naive usa la columna original unidades_hace_7d, sin escalar: no lleva preparacion.
    preparacion = "passthrough" if isinstance(modelo, NaiveEstacional) else nueva_preparacion()
    pipe = Pipeline([("preparacion", preparacion), ("modelo", modelo)])

    with mlflow.start_run(run_name=nombre):
        mlflow.set_tags({"familia": familia, "metrica": "WAPE"})

        # Hiperparametros del modelo
        mlflow.log_params(modelo.get_params())
        # Como se dividieron los datos y con que archivo
        mlflow.log_params(division)
        mlflow.log_params({"datos_archivo": ARCHIVO, "datos_sha256": HUELLA})

        inicio = time.perf_counter()
        pipe.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio

        w_ent = wape(y_ent, pipe.predict(X_ent))
        w_prueba = wape(y_prueba, pipe.predict(X_prueba))
        mlflow.log_metrics({
            "wape_entrenamiento": w_ent,
            "wape_prueba": w_prueba,
            "brecha": w_prueba - w_ent,
            "segundos_entrenamiento": segundos,
        })

        # pipeline completo (preparacion + modelo) como artefacto.
        info = mlflow.sklearn.log_model(pipe, "modelo", signature=firma,
                                        serialization_format="cloudpickle",
                                        code_paths=["pipeline_proyecto.py"])
        URI_MODELO[mlflow.active_run().info.run_id] = info.model_uri

    print(f"{nombre:<41} WAPE ent {w_ent:.4f} | prueba {w_prueba:.4f} | "
          f"brecha {w_prueba - w_ent:+.4f} | {segundos:.1f} s")

Naive estacional (igual que hace 7 dias)  WAPE ent 0.8542 | prueba 0.9001 | brecha +0.0459 | 0.0 s
DummyRegressor (media)                    WAPE ent 1.2146 | prueba 1.2079 | brecha -0.0067 | 1.2 s
Ridge (alpha=1)                           WAPE ent 0.8620 | prueba 0.8731 | brecha +0.0110 | 1.5 s
Ridge (alpha=10000)                       WAPE ent 0.8624 | prueba 0.8727 | brecha +0.0103 | 1.3 s
Random Forest sin regularizar             WAPE ent 0.6869 | prueba 0.8392 | brecha +0.1524 | 45.5 s
Random Forest regularizado                WAPE ent 0.8474 | prueba 0.8554 | brecha +0.0080 | 29.0 s
Red neuronal                              WAPE ent 0.8223 | prueba 0.8434 | brecha +0.0210 | 385.8 s


## 4. Comparación desde el registro

In [6]:
corridas = mlflow.search_runs(experiment_names=[EXPERIMENTO])
columnas = ["run_id", "tags.mlflow.runName", "tags.familia", "metrics.wape_entrenamiento",
            "metrics.wape_prueba", "metrics.brecha", "metrics.segundos_entrenamiento"]
corridas[columnas].sort_values("metrics.wape_prueba").round(4)

,run_id,tags.mlflow.runName,tags.familia,metrics.wape_entrenamiento,metrics.wape_prueba,metrics.brecha,metrics.segundos_entrenamiento
2,f85a7d9db2f44e0896f13b648b67e4e4,Random Forest sin regularizar,ensamblaje,0.6869,0.8392,0.1524,45.5285
0,4f9dec5843854d5e9f82fadf026b75ed,Red neuronal,red_neuronal,0.8223,0.8434,0.0210,385.7616
1,324cbfd6fbaa4167bbaf5b6930a2f1a8,Random Forest regularizado,ensamblaje,0.8474,0.8554,0.0080,29.0031
3,25c49d83e91e47cb898aaa96a5e5738b,Ridge (alpha=10000),lineal,0.8624,0.8727,0.0103,1.2551
4,71d2f52915b948ad99da527f80f20d5e,Ridge (alpha=1),lineal,0.8620,0.8731,0.0110,1.5495
6,8b4528e660af45aa960f80356de2dd0d,Naive estacional (igual que hace 7 dias),linea_base,0.8542,0.9001,0.0459,0.0003
7,3a16eb2239874a1ab92eda89418c1557,Naive estacional (igual que hace 7 dias),linea_base,0.8542,0.9001,0.0459,0.0002
8,81fb0b6750a445918f6074ef1e79ff96,Naive estacional (igual que hace 7 dias),linea_base,0.8542,0.9001,0.0459,0.0003
5,b2d0f0635429493ea29abe3316f220f4,DummyRegressor (media),linea_base,1.2146,1.2079,-0.0067,1.1533


## 5. Corrida final

Se elige la corrida con menor WAPE en prueba.

In [7]:
final = corridas.sort_values("metrics.wape_prueba").iloc[0]
RUN_FINAL = final["run_id"]
print("Corrida final:", final["tags.mlflow.runName"], "| run_id:", RUN_FINAL)

cliente = MlflowClient()
cliente.set_tag(RUN_FINAL, "candidato", "final")

registro = mlflow.register_model(URI_MODELO[RUN_FINAL], "m5-modelo-final")
print("Registrado como m5-modelo-final, version", registro.version)

Corrida final: Random Forest sin regularizar | run_id: f85a7d9db2f44e0896f13b648b67e4e4
Registrado como m5-modelo-final, version 1


Successfully registered model 'm5-modelo-final'.
Created version '1' of model 'm5-modelo-final'.


In [8]:
modelo_cargado = mlflow.sklearn.load_model(f"models:/m5-modelo-final/{registro.version}")

wape_registrado = cliente.get_run(RUN_FINAL).data.metrics["wape_prueba"]
wape_recalculado = wape(y_prueba, modelo_cargado.predict(X_prueba))

print(f"WAPE de prueba registrado:  {wape_registrado:.6f}")
print(f"WAPE de prueba recalculado: {wape_recalculado:.6f}")
assert np.isclose(wape_registrado, wape_recalculado), "El modelo cargado NO reproduce la metrica"
print("OK: el modelo cargado desde el registro reproduce la metrica registrada")

WAPE de prueba registrado:  0.839242
WAPE de prueba recalculado: 0.839242
OK: el modelo cargado desde el registro reproduce la metrica registrada


## 6. Exportar la tabla de corridas

In [ ]:
mlflow.search_runs(experiment_names=[EXPERIMENTO]).to_csv("corridas_proyecto.csv", index=False)
print("Guardado corridas_proyecto.csv")

Guardado corridas_proyecto.csv


: 